<a href="https://colab.research.google.com/github/emaantech99/Electric-Propulsion-optimizer/blob/main/electric_proplusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AUTO size electric propulsion

In [ ]:
import numpy as np

# =====================================================
# USER INPUTS
# =====================================================

weight_kg = float(input("Aircraft Weight (kg): "))
cruise_speed = float(input("Cruise Speed (m/s): "))
target_endurance = float(input("Target Endurance (hr): "))
target_range = float(input("Target Range (km): "))

# =====================================================
# DEFAULT UAV PARAMETERS
# =====================================================

rho = 1.225
g = 9.81

CD0 = 0.025
e = 0.85
AR = 12
CL = 0.8

eta_motor = 0.92
eta_prop = 0.85
eta_ESC = 0.98

eta_total = eta_motor * eta_prop * eta_ESC

# =====================================================
# AERODYNAMIC CALCULATIONS
# =====================================================

S = weight_kg * g / (0.5 * rho * cruise_speed**2 * CL)

K = 1 / (np.pi * e * AR)

CD = CD0 + K * CL**2

Drag = 0.5 * rho * cruise_speed**2 * S * CD

Power_required = Drag * cruise_speed

# Add 30% design margin
Motor_power = Power_required * 1.3

# =====================================================
# BATTERY SIZING
# =====================================================

Energy_required_Wh = (Motor_power * target_endurance) / eta_total

# =====================================================
# VOLTAGE SELECTION
# =====================================================

if weight_kg < 5:
    Battery_voltage = 22.2      # 6S

elif weight_kg < 20:
    Battery_voltage = 44.4      # 12S

elif weight_kg < 50:
    Battery_voltage = 51.8      # 14S

else:
    Battery_voltage = 96.0

Battery_capacity_Ah = Energy_required_Wh / Battery_voltage

# =====================================================
# ESC SIZING
# =====================================================

Current_draw = Motor_power / Battery_voltage

ESC_rating = Current_draw * 1.3

# =====================================================
# PROPELLER SIZING
# =====================================================

# Simple empirical sizing

if weight_kg < 2:
    Prop_diameter = 12

elif weight_kg < 5:
    Prop_diameter = 16

elif weight_kg < 10:
    Prop_diameter = 20

elif weight_kg < 20:
    Prop_diameter = 24

elif weight_kg < 50:
    Prop_diameter = 30

else:
    Prop_diameter = 40

# =====================================================
# VERIFY RANGE
# =====================================================

Estimated_range = cruise_speed * target_endurance * 3600 / 1000

# =====================================================
# RESULTS
# =====================================================

print("\n==============================")
print("AUTO PROPULSION DESIGNER")
print("==============================")

print(f"\nWing Area Required = {S:.2f} m²")

print(f"\nPower Required = {Power_required:.1f} W")

print(f"Recommended Motor Power = {Motor_power:.1f} W")

print(f"\nBattery Voltage = {Battery_voltage:.1f} V")

print(f"Battery Capacity = {Battery_capacity_Ah:.1f} Ah")

print(f"Battery Energy = {Energy_required_Wh:.1f} Wh")

print(f"\nESC Rating = {ESC_rating:.1f} A")

print(f"\nRecommended Propeller Diameter = {Prop_diameter} inch")

print(f"\nEstimated Range = {Estimated_range:.1f} km")

print("\nMission Requirements Satisfied")

OPTIMIZED PROPULSION DESIGNER

In [ ]:
import numpy as np
from scipy.optimize import minimize

# ==========================================
# USER INPUTS
# ==========================================

weight_kg = float(input("Aircraft Weight (kg): "))
cruise_speed = float(input("Cruise Speed (m/s): "))
target_endurance = float(input("Target Endurance (hr): "))
target_range = float(input("Target Range (km): "))

# ==========================================
# DEFAULT UAV PARAMETERS
# ==========================================

rho = 1.225
g = 9.81

CD0 = 0.025
e = 0.85
AR = 12
CL = 0.8

eta_motor = 0.92
eta_prop = 0.85
eta_ESC = 0.98

eta_total = eta_motor * eta_prop * eta_ESC

# ==========================================
# AERODYNAMIC MODEL
# ==========================================

S = weight_kg * g / (0.5 * rho * cruise_speed**2 * CL)

K = 1 / (np.pi * e * AR)

CD = CD0 + K * CL**2

Drag = 0.5 * rho * cruise_speed**2 * S * CD

Power_required = Drag * cruise_speed

# ==========================================
# WEIGHT MODELS
# ==========================================

def battery_weight(voltage, capacity):
    energy_wh = voltage * capacity
    return energy_wh / 220

def motor_weight(power):
    return power / 4000

# ==========================================
# OBJECTIVE FUNCTION
# ==========================================

def objective(x):

    voltage = x[0]
    capacity = x[1]
    motor_power = x[2]
    prop_diameter = x[3]

    W_batt = battery_weight(voltage, capacity)
    W_motor = motor_weight(motor_power)

    return W_batt + W_motor

# ==========================================
# CONSTRAINTS
# ==========================================

def endurance_constraint(x):

    voltage = x[0]
    capacity = x[1]

    energy_wh = voltage * capacity * eta_total

    endurance_hr = energy_wh / Power_required

    return endurance_hr - target_endurance

def range_constraint(x):

    voltage = x[0]
    capacity = x[1]

    energy_wh = voltage * capacity * eta_total

    endurance_hr = energy_wh / Power_required

    range_km = cruise_speed * endurance_hr * 3600 / 1000

    return range_km - target_range

def thrust_constraint(x):

    motor_power = x[2]

    thrust = motor_power / cruise_speed

    TW = thrust / (weight_kg * g)

    return TW - 0.3

def power_constraint(x):

    motor_power = x[2]

    return motor_power - 1.3 * Power_required

constraints = [

    {'type':'ineq','fun':endurance_constraint},

    {'type':'ineq','fun':range_constraint},

    {'type':'ineq','fun':thrust_constraint},

    {'type':'ineq','fun':power_constraint}

]

# ==========================================
# DESIGN VARIABLE BOUNDS
# ==========================================

bounds = [

    (22.2,96.0),      # Voltage

    (5,500),          # Capacity Ah

    (500,50000),      # Motor Power W

    (10,60)           # Prop Diameter inch

]

x0 = [44.4,50,3000,24]

# ==========================================
# RUN OPTIMIZER
# ==========================================

result = minimize(
    objective,
    x0,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

V_opt, Ah_opt, P_opt, D_opt = result.x

# ==========================================
# POST PROCESSING
# ==========================================

battery_energy = V_opt * Ah_opt

battery_weight_opt = battery_weight(V_opt, Ah_opt)

motor_weight_opt = motor_weight(P_opt)

ESC_rating = (P_opt / V_opt) * 1.3

thrust = P_opt / cruise_speed

TW = thrust / (weight_kg * g)

# ==========================================
# RESULTS
# ==========================================

print("\n==============================")
print("OPTIMIZED AUTO PROPULSION DESIGNER")
print("==============================")

print(f"\nBattery Voltage = {V_opt:.1f} V")

print(f"Battery Capacity = {Ah_opt:.1f} Ah")

print(f"Battery Energy = {battery_energy:.1f} Wh")

print(f"Battery Weight = {battery_weight_opt:.2f} kg")

print(f"\nMotor Power = {P_opt:.1f} W")

print(f"Motor Weight = {motor_weight_opt:.2f} kg")

print(f"\nESC Rating = {ESC_rating:.1f} A")

print(f"\nPropeller Diameter = {D_opt:.1f} inch")

print(f"\nThrust-to-Weight Ratio = {TW:.2f}")

print(f"\nTotal Propulsion Weight = "
      f"{battery_weight_opt + motor_weight_opt:.2f} kg")

print("\nOptimization Successful:",
      result.success)

Aircraft Weight (kg): 2
Cruise Speed (m/s): 30
Target Endurance (hr): 0.30
Target Range (km): 3

OPTIMIZED AUTO PROPULSION DESIGNER

Battery Voltage = 22.2 V
Battery Capacity = 5.0 Ah
Battery Energy = 111.0 Wh
Battery Weight = 0.50 kg

Motor Power = 2999.9 W
Motor Weight = 0.75 kg

ESC Rating = 175.7 A

Propeller Diameter = 24.0 inch

Thrust-to-Weight Ratio = 5.10

Total Propulsion Weight = 1.25 kg

Optimization Successful: True


ADVANCED


In [ ]:
import numpy as np

# ==================================================
# USER INPUTS
# ==================================================

weight_kg = float(input("Aircraft Weight (kg): "))
wing_area = float(input("Wing Area (m²): "))
AR = float(input("Aspect Ratio: "))
velocity = float(input("Cruise Velocity (m/s): "))

CL = float(input("Predicted CL: "))
CD = float(input("Predicted CD: "))

target_range = float(input("Target Range (km): "))
target_endurance = float(input("Target Endurance (hr): "))

eta_motor = float(input("Motor Efficiency (0-1): "))
eta_prop = float(input("Propeller Efficiency (0-1): "))
eta_ESC = float(input("ESC Efficiency (0-1): "))

battery_voltage = float(input("Battery Voltage (V): "))

# ==================================================
# CONSTANTS
# ==================================================

rho = 1.225
g = 9.81

# ==================================================
# AERODYNAMIC MODULE
# ==================================================

LD = CL / CD

lift = 0.5 * rho * velocity**2 * wing_area * CL

drag = 0.5 * rho * velocity**2 * wing_area * CD

# ==================================================
# PERFORMANCE MODULE
# ==================================================

power_required = drag * velocity

endurance_sec_required = target_endurance * 3600

range_required = target_range * 1000

# ==================================================
# ELECTRIC PROPULSION MODULE
# ==================================================

eta_total = eta_motor * eta_prop * eta_ESC

energy_required_Wh = (
    power_required * target_endurance
) / eta_total

battery_capacity_Ah = (
    energy_required_Wh
) / battery_voltage

# ==================================================
# MOTOR SIZING
# ==================================================

motor_power = power_required * 1.3

# ==================================================
# ESC SIZING
# ==================================================

motor_current = motor_power / battery_voltage

ESC_rating = motor_current * 1.3

# ==================================================
# PROPELLER SIZING
# ==================================================

if weight_kg < 2:
    prop_diameter = 12

elif weight_kg < 5:
    prop_diameter = 16

elif weight_kg < 10:
    prop_diameter = 20

elif weight_kg < 20:
    prop_diameter = 24

elif weight_kg < 50:
    prop_diameter = 30

else:
    prop_diameter = 40

# ==================================================
# THRUST CALCULATION
# ==================================================

thrust_required = drag * 1.3

thrust_to_weight = (
    thrust_required
    /
    (weight_kg * g)
)

# ==================================================
# OUTPUTS
# ==================================================

print("\n")
print("====================================")
print("AERO EFFICIENT UAV DESIGN REPORT")
print("====================================")

print(f"\nLift = {lift:.2f} N")

print(f"Drag = {drag:.2f} N")

print(f"L/D Ratio = {LD:.2f}")

print(f"\nPower Required = {power_required:.2f} W")

print(f"\nTarget Range = {target_range:.2f} km")

print(f"Target Endurance = {target_endurance:.2f} hr")

print("\n----- Battery -----")

print(
f"Required Energy = {energy_required_Wh:.2f} Wh"
)

print(
f"Battery Voltage = {battery_voltage:.1f} V"
)

print(
f"Battery Capacity = {battery_capacity_Ah:.2f} Ah"
)

print("\n----- Motor -----")

print(
f"Recommended Motor Power = {motor_power:.2f} W"
)

print("\n----- ESC -----")

print(
f"Recommended ESC Rating = {ESC_rating:.2f} A"
)

print("\n----- Propeller -----")

print(
f"Recommended Diameter = {prop_diameter} inch"
)

print("\n----- Aircraft Performance -----")

print(
f"Thrust-to-Weight Ratio = {thrust_to_weight:.2f}"
)

print("\nMission Feasibility Check Complete")

KeyboardInterrupt: Interrupted by user

OPTIMIZED

In [ ]:
import numpy as np
from scipy.optimize import minimize

# =====================================================
# USER INPUTS
# =====================================================

weight_kg = float(input("Aircraft Weight (kg): "))
cruise_speed = float(input("Cruise Speed (m/s): "))
target_range = float(input("Target Range (km): "))
target_endurance = float(input("Target Endurance (hr): "))
LD = float(input("L/D Ratio: "))

eta_motor = float(input("Motor Efficiency (0-1): "))
eta_prop = float(input("Propeller Efficiency (0-1): "))
eta_ESC = float(input("ESC Efficiency (0-1): "))

battery_voltage = float(input("Battery Voltage (V): "))

# =====================================================
# CONSTANTS
# =====================================================

g = 9.81
rho = 1.225

eta_total = eta_motor * eta_prop * eta_ESC

# =====================================================
# PROPULSION MODELS
# =====================================================

def battery_weight(capacity_Ah):

    energy_Wh = battery_voltage * capacity_Ah

    # Li-ion ≈ 220 Wh/kg
    return energy_Wh / 220


def motor_weight(power_W):

    # empirical UAV relation
    return power_W / 4000


def power_required():

    drag = (weight_kg * g) / LD

    return drag * cruise_speed


P_req = power_required()

# =====================================================
# OBJECTIVE FUNCTION
# =====================================================

def objective(x):

    capacity_Ah = x[0]
    motor_power = x[1]

    W_batt = battery_weight(capacity_Ah)
    W_motor = motor_weight(motor_power)

    return W_batt + W_motor


# =====================================================
# CONSTRAINTS
# =====================================================

def endurance_constraint(x):

    capacity_Ah = x[0]

    energy_Wh = (
        battery_voltage
        * capacity_Ah
        * eta_total
    )

    endurance_hr = energy_Wh / P_req

    return endurance_hr - target_endurance


def range_constraint(x):

    capacity_Ah = x[0]

    energy_Wh = (
        battery_voltage
        * capacity_Ah
        * eta_total
    )

    endurance_hr = energy_Wh / P_req

    range_km = (
        cruise_speed
        * endurance_hr
        * 3600
        / 1000
    )

    return range_km - target_range


def thrust_constraint(x):

    motor_power = x[1]

    thrust = motor_power / cruise_speed

    TW = thrust / (weight_kg * g)

    return TW - 0.3


constraints = [

    {'type': 'ineq',
     'fun': endurance_constraint},

    {'type': 'ineq',
     'fun': range_constraint},

    {'type': 'ineq',
     'fun': thrust_constraint}

]

# =====================================================
# INITIAL GUESS
# =====================================================

x0 = [50, 3000]

bounds = [

    (5, 500),       # Battery Ah

    (500, 50000)    # Motor Power W

]

# =====================================================
# OPTIMIZATION
# =====================================================

result = minimize(

    objective,

    x0,

    bounds=bounds,

    constraints=constraints,

    method='SLSQP'

)

# =====================================================
# RESULTS
# =====================================================

capacity_opt = result.x[0]
motor_power_opt = result.x[1]

battery_energy = (
    battery_voltage
    * capacity_opt
)

battery_weight_opt = battery_weight(capacity_opt)

motor_weight_opt = motor_weight(motor_power_opt)

esc_rating = (
    motor_power_opt
    / battery_voltage
) * 1.3

# simple prop sizing
prop_diameter = 0.12 * np.sqrt(motor_power_opt)

print("\n===================================")
print("AUTO PROPULSION DESIGNER")
print("===================================")

print(f"\nBattery Capacity = {capacity_opt:.1f} Ah")

print(f"Battery Voltage = {battery_voltage:.1f} V")

print(f"Battery Energy = {battery_energy:.1f} Wh")

print(f"Battery Weight = {battery_weight_opt:.2f} kg")

print(f"\nMotor Power = {motor_power_opt:.1f} W")

print(f"Motor Weight = {motor_weight_opt:.2f} kg")

print(f"\nESC Rating = {esc_rating:.1f} A")

print(f"\nPropeller Diameter = {prop_diameter:.1f} inch")

print(f"\nTotal Propulsion Weight = "
      f"{battery_weight_opt + motor_weight_opt:.2f} kg")

print("\nMission Requirements Satisfied")

Aircraft Weight (kg): 2.5
Cruise Speed (m/s): 30
Target Range (km): 5
Target Endurance (hr): 0.5
L/D Ratio: 20
Motor Efficiency (0-1): 0.025
Propeller Efficiency (0-1): 0.035
ESC Efficiency (0-1): 0.0045
Battery Voltage (V): 5

AUTO PROPULSION DESIGNER

Battery Capacity = 50.0 Ah
Battery Voltage = 5.0 V
Battery Energy = 250.0 Wh
Battery Weight = 1.14 kg

Motor Power = 500.0 W
Motor Weight = 0.13 kg

ESC Rating = 130.0 A

Propeller Diameter = 2.7 inch

Total Propulsion Weight = 1.26 kg

Mission Requirements Satisfied
